# Lesson 1: 音とは何か・デジタル音の基礎

**コンパニオンノートブック** — 詳しい解説は本文 Lesson 1 を参照してください。

## セットアップ

In [ ]:
# --- 最初に1回だけ実行 ---
import sys
try:
    import google.colab
    !pip install -q japanize-matplotlib
    !git clone -q https://github.com/ggszk/simple-sound-programming.git
    sys.path.append('/content/simple-sound-programming')
except ImportError:
    sys.path.append('..')

from audio_lib.notebook import setup_environment
setup_environment()

## このレッスンで学ぶこと

- 音の物理的性質（振動、周波数、振幅）を理解する
- Pythonでサイン波を生成し、音を鳴らす
- アナログ音声とデジタル音声の違いを理解する
- サンプリングと量子化の仕組みを知る


In [ ]:
import numpy as np
from IPython.display import display
from audio_lib import sine_wave, sawtooth_wave, AudioSignal
from audio_lib.synthesis.note_utils import note_to_frequency, note_name_to_number
from audio_lib.notebook import play_sound, plot_waveform

## 1.2 最初の音を鳴らそう

In [ ]:
# 440Hz（ラの音）のサイン波を1秒間生成
signal = sine_wave(frequency=440, duration=1.0)

In [ ]:
print(f"データの長さ: {signal.num_samples} サンプル")
print(f"サンプリングレート: {signal.sample_rate} Hz")
print(f"長さ: {signal.duration} 秒")

### 波形を見る

In [ ]:
plot_waveform(signal, duration=0.01, title="440Hz サイン波")

### 音を再生する

In [ ]:
display(play_sound(signal, "440Hz サイン波（ラの音）"))

## 1.3 中身を見てみよう — サイン波の原理

In [ ]:
sample_rate = 44100          # 1秒あたりのサンプル数
duration = 1.0               # 長さ（秒）
frequency = 440              # 周波数（Hz）

# 時間軸の配列を作る
t = np.linspace(0, duration, int(sample_rate * duration), endpoint=False)

# サイン波の数式をそのまま書く
data = np.sin(2 * np.pi * frequency * t)

### audio_lib との対応

In [ ]:
signal_manual = AudioSignal(data, sample_rate)

### サンプリングレートの違いを体験する

In [ ]:
# CD品質（44100Hz）
saw_cd = sawtooth_wave(440, 1.5, sample_rate=44100)

# 電話品質（8000Hz）
saw_tel = sawtooth_wave(440, 1.5, sample_rate=8000)

display(play_sound(saw_cd, "CD品質 (44100Hz) ノコギリ波"))
display(play_sound(saw_tel, "電話品質 (8000Hz) ノコギリ波"))

### エイリアシング

In [ ]:
# 5000Hzの音を異なるサンプリングレートで記録
high_cd  = sine_wave(5000, 0.5, sample_rate=44100)   # 正確に記録できる
high_tel = sine_wave(5000, 0.5, sample_rate=8000)     # エイリアシング発生

display(play_sound(high_cd,  "5000Hz（CD品質: 正確）"))
display(play_sound(high_tel, "5000Hz（電話品質: エイリアシング）"))

## 1.6 周波数と音の高さ

In [ ]:
frequencies = [220, 440, 880, 1760]

for freq in frequencies:
    sig = sine_wave(freq, 1.0)
    display(play_sound(sig, f"{freq}Hz"))

## 1.7 振幅と音の大きさ

In [ ]:
volumes = [0.1, 0.3, 0.8, 1.0]

for vol in volumes:
    sig = sine_wave(440, 1.0) * vol
    display(play_sound(sig, f"440Hz 音量={vol}"))

### audio_lib で保存

In [ ]:
signal = sine_wave(440, 2.0)
signal.save("my_sine.wav")

### 素のPythonで保存

In [ ]:
from scipy.io import wavfile

sample_rate = 44100
data = np.sin(2 * np.pi * 440 * np.linspace(0, 2.0, 44100 * 2, endpoint=False))

# -1.0〜1.0 の float を 16bit 整数に変換
audio_16bit = (data * 32767).astype(np.int16)

wavfile.write("my_sine_raw.wav", sample_rate, audio_16bit)

## 1.9 音名と周波数

In [ ]:
note_names = ["C4", "D4", "E4", "F4", "G4", "A4", "B4", "C5"]
jp_names   = ["ド", "レ", "ミ", "ファ", "ソ", "ラ", "シ", "ド"]

print("音名  | 日本名 | MIDI番号 | 周波数 (Hz)")
print("-" * 45)
# zip() は2つのリストから要素を1つずつ取り出してペアにする関数
for name, jp in zip(note_names, jp_names):
    midi_num = note_name_to_number(name)
    freq = note_to_frequency(midi_num)
    print(f" {name}  |  {jp:2s}  |    {midi_num}    | {freq:.2f}")